# Flight Trajectory Prediction Results

This notebook evaluates the performance of the BN flight trajectory prediction model through various metrics, visualizations, and calibration analysis.

## Setup and Configuration

In [ ]:
# Core imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import pathlib
from tqdm.auto import tqdm
import importlib

# Add project root to Python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# External libraries
from traffic.core import Traffic
import pyproj
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project modules
import utils
importlib.reload(utils)

from utils import (
    cache_paths,
    denorm_seq_to_global,
    ResidualGaussianBN,
    sample_many_bn,
    pit_values,
    ade_fde,
    energy_score_per_horizon
)

# Configuration
PARQUET_PATH = "./dataset_cache/traffic_enroute_filtered.parquet"
CACHE_DIR = pathlib.Path("./dataset_cache")
CKPT_PATH = "./models/bn_improved.pt"
CACHE_KEY_FILE = None # Let it find the latest one
OUTPUT_STRIDE_SECONDS = 5
TEMPERATURE = 0.4 # For sampling

# Device configuration
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


## Data Loading

In [205]:
# Load cached dataset artifacts
def load_cache_key():
    """Load dataset cache key file."""
    if CACHE_KEY_FILE:
        key_path = Path(CACHE_KEY_FILE)
        if not key_path.is_absolute():
            key_path = CACHE_DIR / key_path
        if not key_path.exists():
            raise FileNotFoundError(
                f"Specified cache key file not found: {key_path}. "
                f"Make sure it exists under {CACHE_DIR}."
            )
    else:
        key_files = sorted(
            CACHE_DIR.glob("*.key.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if not key_files:
            raise FileNotFoundError(
                "No dataset_cache key files found. Run the training notebook first."
            )
        key_path = key_files[0]
    return key_path

# Load cache key and dataset information
key_path = load_cache_key()
key_info = json.loads(key_path.read_text())
print(f"[cache] using key: {key_path.name}")

dset_key = key_info["dataset_key"]
stats_key = key_info["stats_key"]
paths = cache_paths(dset_key, stats_key)

# Load dataset arrays (using memory-mapped files for efficiency)
X_train = np.load(paths["x_tr"], mmap_mode="r")
Y_train = np.load(paths["y_tr"], mmap_mode="r")
C_train = np.load(paths["c_tr"], mmap_mode="r")
X_val = np.load(paths["x_va"], mmap_mode="r")
Y_val = np.load(paths["y_va"], mmap_mode="r")
C_val = np.load(paths["c_va"], mmap_mode="r")
X_test = np.load(paths["x_te"], mmap_mode="r")
Y_test = np.load(paths["y_te"], mmap_mode="r")
C_test = np.load(paths["c_te"], mmap_mode="r")

# Load normalization statistics and metadata
norm_stats = json.loads(paths["stats"].read_text())
meta_train = pd.read_parquet(paths["meta_tr"])  # per-window metadata
meta_val = pd.read_parquet(paths["meta_va"])
meta_test = pd.read_parquet(paths["meta_te"])
manifest = json.loads(paths["manifest"].read_text())
summary = json.loads(paths["summary"].read_text())

# Extract normalization parameters
feat_mean = norm_stats["feat_mean"]
feat_std = norm_stats["feat_std"]
ctx_mean = norm_stats["ctx_mean"]
ctx_std = norm_stats["ctx_std"]

# Display dataset sizes
dataset_sizes = {
    k: (int(v) if isinstance(v, (int, np.integer)) else v)
    for k, v in summary.get("sizes", {}).items()
}
print("Dataset sizes:", dataset_sizes)

# Load trajectory data
trajs = Traffic.from_file(PARQUET_PATH)
print(f"Loaded {trajs.data.flight_id.nunique()} trajectories from {PARQUET_PATH}")

[cache] using key: 51397c8e8791f8ca.key.json
Dataset sizes: {'train': 2000000, 'val': 500000, 'test': 400000}
Loaded 1000 trajectories from ./dataset_cache/traffic_enroute_filtered.parquet


## Model Loading

In [195]:
# Initialize and load the BN model
model = ResidualGaussianBN(state_dim=7, context_dim=8, hidden=512).to(DEVICE)
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Loaded BN model from: {CKPT_PATH}")

Loaded BN model from: ./models/bn_definitive_v2.pt


## Sample Generation and Visualization

Generate trajectory predictions and create visualizations.

In [ ]:
# Choose trajectories for analysis
CASE_LIST = np.random.choice(len(X_test), size=3, replace=False)
N_TRAJS = len(CASE_LIST)
N_SAMPLES = 100
TIMESTEPS = int(Y_test.shape[1])

# Prepare input data
x_hist_batch = torch.from_numpy(np.array([X_test[idx] for idx in CASE_LIST])).to(DEVICE).contiguous().float()
ctx_batch = torch.from_numpy(np.array([C_test[idx] for idx in CASE_LIST])).to(DEVICE).contiguous().float()

print(f"Processing {N_TRAJS} trajectories with {N_SAMPLES} samples each")
print(f"Context shape: {ctx_batch.shape}")

# Generate future trajectory samples
y_norm_all = sample_many_bn(
    model,
    x_hist_batch,
    ctx_batch,
    T_out=TIMESTEPS,
    n_samples=N_SAMPLES,
    temperature=TEMPERATURE
)

# Convert back to global coordinates
y_glob_all = (
    denorm_seq_to_global(
        y_norm_all.reshape(-1, TIMESTEPS, 7),
        ctx_batch.repeat(N_SAMPLES, 1),
        feat_mean, feat_std, ctx_mean, ctx_std,
    )
    .cpu()
    .numpy()
    .reshape(N_SAMPLES, N_TRAJS, TIMESTEPS, -1)
)

# Get history and ground truth in global coordinates
x_hist_glob = (
    denorm_seq_to_global(
        x_hist_batch, ctx_batch, feat_mean, feat_std, ctx_mean, ctx_std
    )
    .cpu()
    .numpy()[:, :, :3]
)

y_true_glob = (
    denorm_seq_to_global(
        torch.from_numpy(np.array([Y_test[idx] for idx in CASE_LIST])).to(DEVICE).float(),
        ctx_batch, feat_mean, feat_std, ctx_mean, ctx_std,
    )
    .cpu()
    .numpy()[:, :, :3]
)

print(f"Generated predictions with shape: {y_glob_all.shape}")
print(f"History shape: {x_hist_glob.shape}, Ground truth shape: {y_true_glob.shape}")

Processing 3 trajectories with 1000 samples each
Context shape: torch.Size([3, 8])
Generated predictions with shape: (1000, 3, 12, 7)
History shape: (3, 60, 3), Ground truth shape: (3, 12, 3)


## Coordinate System Conversion

Convert between different coordinate systems for visualization.

In [197]:
# Coordinate transformation setup
crs_lv95 = pyproj.CRS.from_epsg(2056)  # Swiss LV95
crs_wgs84 = pyproj.CRS.from_epsg(4326)  # WGS84 (lat/lon)
to_wgs84 = pyproj.Transformer.from_crs(crs_lv95, crs_wgs84, always_xy=True)

def xy_to_lonlat_arr(xy: np.ndarray) -> np.ndarray:
    """Convert XY coordinates to longitude/latitude."""
    lons, lats = to_wgs84.transform(xy[:, 0], xy[:, 1])
    return np.stack([lons, lats], axis=1)

def concat_polyline_lonlat(list_of_ll_arrays):
    """Concatenate multiple lon/lat arrays with None separators for Plotly."""
    lons, lats = [], []
    for arr in list_of_ll_arrays:
        lons.extend(arr[:, 0].tolist())
        lats.extend(arr[:, 1].tolist())
        lons.append(None)
        lats.append(None)
    if lons: lons.pop()  # Remove last None
    if lats: lats.pop()  # Remove last None
    return lons, lats

## Spaghetti Plot Visualization

Create spaghetti plots showing multiple trajectory predictions.

In [198]:
# Color scheme for plots
COLOR_HISTORY = "black"
COLOR_GT = "red"
COLOR_PRED = "#1f77b4"
COLOR_MEAN = "#e19f20"

# Create subplot grid
cols = min(3, N_TRAJS)
fig_spaghetti = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "map"} for _ in range(3)]],
    subplot_titles=[
        f"{meta_test.loc[CASE_LIST[i]]['flight_id'].split('_')[1]}" if i < N_TRAJS else "—"
        for i in range(3)
    ],
    horizontal_spacing=0.02,
)

for b in range(cols):
    # Convert coordinates to lon/lat
    hist_xy = x_hist_glob[b, :, :2]
    true_xy = y_true_glob[b, :, :2]
    samples_xy = [y_glob_all[s, b, :, :2] for s in range(N_SAMPLES)]

    hist_ll = xy_to_lonlat_arr(hist_xy)
    true_ll = xy_to_lonlat_arr(true_xy)
    samples_ll = [xy_to_lonlat_arr(sxy) for sxy in samples_xy]

    # Center map on trajectory
    all_ll = np.vstack([hist_ll, true_ll])
    lon_center = float((np.nanmin(all_ll[:, 0]) + np.nanmax(all_ll[:, 0])) * 0.5)
    lat_center = float((np.nanmin(all_ll[:, 1]) + np.nanmax(all_ll[:, 1])) * 0.5)

    # Concatenate samples into one polyline trace
    samp_lon, samp_lat = concat_polyline_lonlat(samples_ll)

    showleg = (b == 0)

    # Predicted samples (blue, semi-transparent spaghetti)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=samp_lon, lat=samp_lat,
            mode="lines",
            line=dict(width=1, color=COLOR_PRED),
            opacity=0.5,
            name=f"{N_SAMPLES} samples", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Sample mean (orange)
    mean_ll = np.mean(np.stack(samples_ll, axis=0), axis=0)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=mean_ll[:, 0], lat=mean_ll[:, 1],
            mode="lines",
            line=dict(width=3, color=COLOR_MEAN),
            name="Sample mean", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # History (black)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=hist_ll[:, 0], lat=hist_ll[:, 1],
            mode="lines",
            line=dict(width=1.5, color=COLOR_HISTORY),
            name="History", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Ground Truth (red, on top)
    fig_spaghetti.add_trace(
        go.Scattermap(
            lon=true_ll[:, 0], lat=true_ll[:, 1],
            mode="lines+markers",
            line=dict(width=1, color=COLOR_GT),
            marker=dict(size=6, color=COLOR_GT),
            name="Ground Truth", showlegend=showleg
        ),
        row=1, col=b+1
    )

    # Map settings
    key = "" if b == 0 else str(b+1)
    fig_spaghetti.update_layout({
        f"map{key}": dict(
            style="carto-positron",
            center=dict(lon=lon_center, lat=lat_center),
            zoom=10.5,
        )
    })

# Fill empty panels if fewer than 3
for k in range(2, 4):
    if k > cols:
        fig_spaghetti.update_layout({f"map{k}": dict(style="carto-positron")})

fig_spaghetti.update_layout(
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

# display
fig_spaghetti.show()

In [203]:
# Create Altitude (Z) vs X visualization
fig_z = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"{meta_test.loc[CASE_LIST[i]]['flight_id'].split('_')[1]}" if i < N_TRAJS else "—"
        for i in range(3)
    ],
    horizontal_spacing=0.05,
    shared_yaxes=True
)

for b in range(cols):
    # History
    x_coords_hist = x_hist_glob[b, :, 0]
    z_hist = x_hist_glob[b, :, 2]
    fig_z.add_trace(
        go.Scatter(
            x=x_coords_hist, y=z_hist,
            mode="lines",
            line=dict(color=COLOR_HISTORY, width=2),
            name="History", showlegend=(b == 0)
        ),
        row=1, col=b+1
    )

    # Ground Truth
    x_coords_true = y_true_glob[b, :, 0]
    true_z = y_true_glob[b, :, 2]
    fig_z.add_trace(
        go.Scatter(
            x=x_coords_true, y=true_z,
            mode="lines+markers",
            line=dict(color=COLOR_GT, width=2),
            marker=dict(size=4),
            name="Ground Truth", showlegend=(b == 0)
        ),
        row=1, col=b+1
    )

    # Samples (Blue)
    samples_x = y_glob_all[:, b, :, 0] # (N_SAMPLES, TIMESTEPS)
    samples_z = y_glob_all[:, b, :, 2] # (N_SAMPLES, TIMESTEPS)
    
    # Take a subset if N_SAMPLES is very large
    max_sample_traces = 100
    sample_idx = np.random.choice(N_SAMPLES, size=min(N_SAMPLES, max_sample_traces), replace=False)
    
    for s in sample_idx:
        fig_z.add_trace(
            go.Scatter(
                x=samples_x[s], y=samples_z[s],
                mode="lines",
                line=dict(color=COLOR_PRED, width=1),
                opacity=0.1,
                showlegend=False
            ),
            row=1, col=b+1
        )
    
    # Legend proxy for samples
    fig_z.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode="lines",
            line=dict(color=COLOR_PRED, width=1),
            name=f"Samples (subset of {N_SAMPLES})", showlegend=(b == 0)
        ),
        row=1, col=b+1
    )

    # Sample Mean
    mean_x = np.mean(samples_x, axis=0)
    mean_z = np.mean(samples_z, axis=0)
    fig_z.add_trace(
        go.Scatter(
            x=mean_x, y=mean_z,
            mode="lines",
            line=dict(color=COLOR_MEAN, width=3),
            name="Sample mean", showlegend=(b == 0)
        ),
        row=1, col=b+1
    )

    fig_z.update_xaxes(title_text="X [m]", row=1, col=b+1)
    if b == 0:
        fig_z.update_yaxes(title_text="Altitude Z [m]", row=1, col=b+1)

fig_z.update_layout(
    height=400, width=1100,
    template="plotly_white",
    margin=dict(l=50, r=10, t=50, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.25,
        xanchor="center", x=0.5
    )
)

fig_z.show()

## Model Quality Metrics

Evaluate model performance using RMSE and MAE metrics.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def constant_velocity_extrapolation(
    X_test,
    C_test,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device,
    T_out: int,
    dt_seconds: float = OUTPUT_STRIDE_SECONDS,
):
    """Deterministic constant-velocity baseline in global coordinates."""
    # Denorm history to global to read last state
    x_hist = torch.from_numpy(np.array(X_test)).to(device).contiguous().float()
    ctx = torch.from_numpy(np.array(C_test)).to(device).contiguous().float()
    x_hist_glob = (
        denorm_seq_to_global(x_hist, ctx, feat_mean, feat_std, ctx_mean, ctx_std)
        .cpu()
        .numpy()
    )  # (N, L, D)

    N, L, D = x_hist_glob.shape
    pos0 = x_hist_glob[:, -1, :3]  # (N, 3): x,y,z at last history step
    vel0 = np.zeros((N, 3), dtype=pos0.dtype)
    if D >= 6:
        vel0 = x_hist_glob[:, -1, 3:6]  # (N, 3): vx,vy,vz in m/s

    # Prepare output (N, T_out, D)
    y_pred_glob = np.zeros((N, T_out, D), dtype=x_hist_glob.dtype)

    # Fill positions by extrapolation, velocities constant; other channels set to 0
    tgrid = np.arange(1, T_out + 1, dtype=pos0.dtype) * dt_seconds  # seconds ahead
    disp = tgrid[None, :, None] * vel0[:, None, :]  # (N,T,3)

    # Positions
    y_pred_glob[:, :, 0] = pos0[:, 0][:, None] + disp[:, :, 0]  # x
    y_pred_glob[:, :, 1] = pos0[:, 1][:, None] + disp[:, :, 1]  # y
    if D >= 3:
        y_pred_glob[:, :, 2] = pos0[:, 2][:, None] + disp[:, :, 2]  # z

    # Velocities
    if D >= 4:
        y_pred_glob[:, :, 3] = vel0[:, 0][:, None]  # vx
    if D >= 5:
        y_pred_glob[:, :, 4] = vel0[:, 1][:, None]  # vy
    if D >= 6:
        y_pred_glob[:, :, 5] = vel0[:, 2][:, None]  # vz

    # Any remaining channels set to 0
    if D > 6:
        y_pred_glob[:, :, 6:] = 0.0

    return y_pred_glob  # global coords

@torch.no_grad()
def rmse_mae_3d_2d_vert_vs_horizon(
    model,
    X,
    Y,
    C,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device,
    n_eval: int = 2000,
    n_samples: int = 32,
    batch_size: int = 128,
    dt_seconds: float = OUTPUT_STRIDE_SECONDS,
    progress: bool = True,
    desc: str = "Evaluating batches",
):
    """Single pass evaluator for RMSE & MAE vs horizon."""
    N = min(int(X.shape[0]), int(n_eval))
    T = int(Y.shape[1])

    def zeros():
        return np.zeros(T, dtype=np.float64)

    # Accumulators
    acc = {
        "rmse": {
            "3d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "2d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "vert": {"mean": zeros(), "best": zeros(), "cv": zeros()},
        },
        "mae": {
            "3d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "2d": {"mean": zeros(), "best": zeros(), "cv": zeros()},
            "vert": {"mean": zeros(), "best": zeros(), "cv": zeros()},
        },
    }
    num_cases = 0

    # Progress tracking
    num_batches = (N + batch_size - 1) // batch_size
    iterator = range(0, N, batch_size)
    if progress:
        iterator = tqdm(iterator, total=num_batches, desc=desc, leave=True)

    for i0 in iterator:
        i1 = min(N, i0 + batch_size)

        # Tensors on device
        x_hist = torch.from_numpy(np.array(X[i0:i1])).to(device).contiguous().float()
        ctx = torch.from_numpy(np.array(C[i0:i1])).to(device).contiguous().float()
        y_true = torch.from_numpy(np.array(Y[i0:i1])).to(device).contiguous().float()
        B = int(x_hist.shape[0])

        # Samples (normalized) → global
        y_norm_all = sample_many_bn(
            model, x_hist, ctx,
            T_out=T, n_samples=n_samples,
            temperature=TEMPERATURE
        )  # (S, B, T, D_norm)

        y_s_glob = denorm_seq_to_global(
            y_norm_all.reshape(-1, T, 7), 
            ctx.repeat(n_samples, 1),
            feat_mean, feat_std, ctx_mean, ctx_std
        ).view(n_samples, B, T, -1)  # (S,B,T,D)

        # Ground truth global
        y_true_glob = denorm_seq_to_global(
            y_true, ctx, feat_mean, feat_std, ctx_mean, ctx_std
        )  # (B,T,D)

        # Mean prediction across samples
        y_mean_glob = y_s_glob.mean(dim=0)  # (B,T,D)

        # Errors for mean
        dx_mean = y_mean_glob[..., 0] - y_true_glob[..., 0]  # (B,T)
        dy_mean = y_mean_glob[..., 1] - y_true_glob[..., 1]
        dz_mean = y_mean_glob[..., 2] - y_true_glob[..., 2]

        dist3_mean_sq = dx_mean**2 + dy_mean**2 + dz_mean**2
        dist2_mean_sq = dx_mean**2 + dy_mean**2
        vert_mean_sq = dz_mean**2

        dist3_mean = torch.sqrt(dist3_mean_sq + 1e-12)
        dist2_mean = torch.sqrt(dist2_mean_sq + 1e-12)
        vert_mean = torch.sqrt(vert_mean_sq + 1e-12)  # = |dz|

        # Errors for best-of-N
        dx_all = y_s_glob[..., 0] - y_true_glob.unsqueeze(0)[..., 0]  # (S,B,T)
        dy_all = y_s_glob[..., 1] - y_true_glob.unsqueeze(0)[..., 1]
        dz_all = y_s_glob[..., 2] - y_true_glob.unsqueeze(0)[..., 2]

        dist3_all_sq = dx_all**2 + dy_all**2 + dz_all**2       # (S,B,T)
        dist2_all_sq = dx_all**2 + dy_all**2                   # (S,B,T)
        vert_all_sq = dz_all**2                               # (S,B,T)

        # Min over samples (S) per (B,T)
        dist3_best_sq = dist3_all_sq.min(dim=0).values
        dist2_best_sq = dist2_all_sq.min(dim=0).values
        vert_best_sq = vert_all_sq.min(dim=0).values

        dist3_best = torch.sqrt(dist3_best_sq + 1e-12)
        dist2_best = torch.sqrt(dist2_best_sq + 1e-12)
        vert_best = torch.sqrt(vert_best_sq + 1e-12)  # = min |dz|

        # Constant-velocity baseline
        cv_pred_glob = constant_velocity_extrapolation(
            X[i0:i1], C[i0:i1],
            feat_mean, feat_std, ctx_mean, ctx_std,
            device=device, T_out=T, dt_seconds=dt_seconds,
        )  # (B,T,D)
        cv = torch.from_numpy(cv_pred_glob).to(device)

        dx_cv = cv[..., 0] - y_true_glob[..., 0]
        dy_cv = cv[..., 1] - y_true_glob[..., 1]
        dz_cv = cv[..., 2] - y_true_glob[..., 2]

        dist3_cv_sq = dx_cv**2 + dy_cv**2 + dz_cv**2
        dist2_cv_sq = dx_cv**2 + dy_cv**2
        vert_cv_sq = dz_cv**2

        dist3_cv = torch.sqrt(dist3_cv_sq + 1e-12)
        dist2_cv = torch.sqrt(dist2_cv_sq + 1e-12)
        vert_cv = torch.sqrt(vert_cv_sq + 1e-12)  # = |dz|

        # Accumulate (sum over batch; average over num_cases later)
        for key, sq_t, abs_t in [
            (("3d","mean"), dist3_mean_sq, dist3_mean),
            (("2d","mean"), dist2_mean_sq, dist2_mean),
            (("vert","mean"), vert_mean_sq, vert_mean),
            (("3d","best"), dist3_best_sq, dist3_best),
            (("2d","best"), dist2_best_sq, dist2_best),
            (("vert","best"), vert_best_sq, vert_best),
            (("3d","cv"), dist3_cv_sq, dist3_cv),
            (("2d","cv"), dist2_cv_sq, dist2_cv),
            (("vert","cv"), vert_cv_sq, vert_cv),
        ]:
            cat, est = key
            acc["rmse"][cat][est] += sq_t.detach().cpu().numpy().sum(axis=0)
            acc["mae"][cat][est]  += abs_t.detach().cpu().numpy().sum(axis=0)

        num_cases += B

        # Progress update
        if progress:
            iterator.set_postfix_str(f"cases={num_cases}, B={B}")

    # Finalize
    denom = max(1, num_cases)
    out = {"horizons_s": (np.arange(1, T + 1) * dt_seconds).astype(int), "rmse": {}, "mae": {}, "fde": {}}
    for cat in ["3d", "2d", "vert"]:
        out["rmse"][cat] = {}
        out["mae"][cat]  = {}
        out["fde"][cat]  = {}
        for est in ["mean", "best", "cv"]:
            rmse_curve = np.sqrt(acc["rmse"][cat][est] / denom)
            mae_curve  = acc["mae"][cat][est] / denom
            out["rmse"][cat][est] = rmse_curve
            out["mae"][cat][est]  = mae_curve
            out["fde"][cat][est] = {
                "rmse": rmse_curve[-1],
                "mae":  mae_curve[-1],
            }
    return out

# Run evaluation metrics
res = rmse_mae_3d_2d_vert_vs_horizon(
    model,
    X_test, Y_test, C_test,
    feat_mean, feat_std, ctx_mean, ctx_std,
    device=DEVICE,
    n_eval=512,  
    n_samples=128,
    batch_size=64,
    dt_seconds=OUTPUT_STRIDE_SECONDS
)

print("Evaluation completed successfully")

Evaluating batches:   0%|          | 0/80 [00:00<?, ?it/s]

Evaluation completed successfully


In [132]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

H = res["horizons_s"]

# --- build 2-column × 3-row subplot ---
fig = make_subplots(
    rows=3, cols=2,
    shared_xaxes=True,
    subplot_titles=[
        "MAE — 3D (x,y,z)", "RMSE — 3D (x,y,z)",
        "MAE — 2D horizontal (x,y)", "RMSE — 2D horizontal (x,y)",
        "MAE — Vertical (z)", "RMSE — Vertical (z)"
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

# === color scheme ===
COLOR_MEAN = "blue"
COLOR_BEST = "green"
COLOR_CV   = "red"

def add_metric_traces(cat_name, row_mae, row_rmse, show_legend=False):
    # --- MAE (left) ---
    fig.add_trace(go.Scatter(
        x=H, y=res["mae"][cat_name]["mean"],
        name="Model (mean)", mode="lines+markers",
        line=dict(width=2, color=COLOR_MEAN),
        showlegend=show_legend
    ), row=row_mae, col=1)

    fig.add_trace(go.Scatter(
        x=H, y=res["mae"][cat_name]["best"],
        name="Model (best-of-S)", mode="lines+markers",
        line=dict(width=2, dash="dash", color=COLOR_BEST),
        showlegend=show_legend
    ), row=row_mae, col=1)

    fig.add_trace(go.Scatter(
        x=H, y=res["mae"][cat_name]["cv"],
        name="Constant velocity", mode="lines+markers",
        line=dict(width=2, dash="dot", color=COLOR_CV),
        showlegend=show_legend
    ), row=row_mae, col=1)

    # --- RMSE (right) ---
    fig.add_trace(go.Scatter(
        x=H, y=res["rmse"][cat_name]["mean"],
        name="Model (mean)", mode="lines+markers",
        line=dict(width=2, color=COLOR_MEAN),
        showlegend=False
    ), row=row_rmse, col=2)

    fig.add_trace(go.Scatter(
        x=H, y=res["rmse"][cat_name]["best"],
        name="Model (best-of-S)", mode="lines+markers",
        line=dict(width=2, dash="dash", color=COLOR_BEST),
        showlegend=False
    ), row=row_rmse, col=2)

    fig.add_trace(go.Scatter(
        x=H, y=res["rmse"][cat_name]["cv"],
        name="Constant velocity", mode="lines+markers",
        line=dict(width=2, dash="dot", color=COLOR_CV),
        showlegend=False
    ), row=row_rmse, col=2)

# Add traces — legend only once (for first row)
add_metric_traces("3d",   1, 1, show_legend=True)
add_metric_traces("2d",   2, 2, show_legend=False)
add_metric_traces("vert", 3, 3, show_legend=False)

# --- layout ---
fig.update_layout(
    template="plotly_white",
    height=900, width=1100,
    # title="MAE and RMSE vs Prediction Horizon",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.1,
        xanchor="center", x=0.5,
    )
)

# axis labels
for r in range(1, 4):
    fig.update_xaxes(title_text="Horizon [s]", row=r, col=1)
    fig.update_xaxes(title_text="Horizon [s]", row=r, col=2)
    fig.update_yaxes(title_text="Error [m]", row=r, col=1)
    fig.update_yaxes(title_text="Error [m]", row=r, col=2)

fig.show()

In [ ]:
# Print ADE and FDE metrics
print(f"Metrics (at {res['horizons_s'][-1]}s horizon):")
for cat in ["3d", "2d", "vert"]:
    print(f"  {cat}:")
    for est in ["mean", "best", "cv"]:
        fde_mae = res["fde"][cat][est]["mae"]
        fde_rmse = res["fde"][cat][est]["rmse"]
        ade_mae = res["mae"][cat][est].mean()
        ade_rmse = res["rmse"][cat][est].mean()
        print(f"    {est:4}: ADE_MAE={ade_mae:7.2f}m, FDE_MAE={fde_mae:7.2f}m")

Metrics (at 60s horizon):
  3d:
    mean: ADE_MAE= 155.73m, FDE_MAE= 399.38m
    best: ADE_MAE=  92.53m, FDE_MAE= 245.45m
    cv  : ADE_MAE= 243.15m, FDE_MAE= 564.54m
  2d:
    mean: ADE_MAE= 149.11m, FDE_MAE= 385.73m
    best: ADE_MAE=  84.18m, FDE_MAE= 227.72m
    cv  : ADE_MAE= 238.25m, FDE_MAE= 553.59m
  vert:
    mean: ADE_MAE=  20.20m, FDE_MAE=  45.10m
    best: ADE_MAE=   8.09m, FDE_MAE=  17.96m
    cv  : ADE_MAE=  21.13m, FDE_MAE=  47.80m


## Probabilistic Calibration Analysis

Analyze the calibration of probabilistic predictions using PIT histograms.

In [ ]:
@torch.no_grad()
def collect_samples_for_calibration(
    model,
    X,
    Y,
    C,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device,
    n_eval: int = 2000,
    n_samples: int = 64,
    batch_size: int = 128,
):
    """Draw samples and return predictions and ground truth."""
    N = min(int(X.shape[0]), int(n_eval))
    T = int(Y.shape[1])
    pos_samples = []
    pos_truth = []
    for i0 in tqdm(range(0, N, batch_size)):
        i1 = min(N, i0 + batch_size)
        x_hist = torch.from_numpy(np.array(X[i0:i1])).to(device).contiguous().float()
        ctx = torch.from_numpy(np.array(C[i0:i1])).to(device).contiguous().float()
        y_true = torch.from_numpy(np.array(Y[i0:i1])).to(device).contiguous().float()
        B = int(x_hist.shape[0])
        
        y_norm_all = sample_many_bn(
            model,
            x_hist,
            ctx,
            T_out=T,
            n_samples=n_samples,
            temperature=TEMPERATURE
        )  # (S, B, T, Dn)
        
        y_glob_all = denorm_seq_to_global(
            y_norm_all.reshape(-1, T, 7), 
            ctx.repeat(n_samples, 1), 
            feat_mean, feat_std, ctx_mean, ctx_std
        ).view(n_samples, B, T, -1)[..., :3]  # (S,B,T,3)
        
        y_true_g = denorm_seq_to_global(
            y_true, ctx, feat_mean, feat_std, ctx_mean, ctx_std
        )[..., :3]
        
        pos_samples.append(y_glob_all)
        pos_truth.append(y_true_g)
        
    ysamp_g = torch.cat(pos_samples, dim=1)  # (S,N,T,3)
    ytrue_g = torch.cat(pos_truth, dim=0)  # (N,T,3)
    return ysamp_g, ytrue_g

# Draw samples for calibration analysis
ysamp_g, ytrue_g = collect_samples_for_calibration(
    model,
    X_test,
    Y_test,
    C_test,
    feat_mean,
    feat_std,
    ctx_mean,
    ctx_std,
    device=DEVICE,
    n_eval=512,
    n_samples=128,
    batch_size=64,
)

print(f"Collected {ysamp_g.shape[0]} samples for {ysamp_g.shape[1]} trajectories")

  0%|          | 0/16 [00:00<?, ?it/s]

Collected 512 samples for 1000 trajectories


## PIT Histogram Visualization

Create Probability Integral Transform (PIT) histograms to assess calibration.

In [ ]:
# Calculate PIT values
pits_btd = pit_values(ysamp_g, ytrue_g)  # (N,T,3)
pits_flat = pits_btd.reshape(-1, 3).detach().cpu().numpy()

# Setup for plotting
axis_names = ["x", "y", "z"]
axis_colors = ["black", "red", "blue"]
uniform_line_color = "gray"

# Create subplots: 1 row x 3 cols (histograms)
fig = make_subplots(
    rows=1, cols=3,
    shared_xaxes=False,
    shared_yaxes=False,
    horizontal_spacing=0.07,
    subplot_titles=[f"PIT histogram — {axis}" for axis in axis_names],
)

# Parameters
nbins = 30

for d in range(3):
    col = d + 1
    color = axis_colors[d]
    u = pits_flat[:, d]

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=u,
            nbinsx=nbins,
            histnorm="probability density",
            marker=dict(color=color),
            name=axis_names[d],
            showlegend=False,
            opacity=0.75,
        ),
        row=1, col=col
    )

    # Uniform PDF reference line (y=1 on [0,1])
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[1, 1],
            mode="lines",
            line=dict(dash="dash", color=uniform_line_color),
            name="Uniform PDF (y=1)" if d == 0 else None,
            showlegend=(d == 0),
        ),
        row=1, col=col
    )

    # Axes formatting
    fig.update_xaxes(title_text="PIT value", range=[0, 1], row=1, col=col)
    fig.update_yaxes(title_text="Density", range=[0, 4.5], row=1, col=col)

# Layout
fig.update_layout(
    height=400, width=800,
    template="plotly_white",
    title="Calibration diagnostics: PIT histograms",
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.3,
        xanchor="center", x=0.5
    ),
    margin=dict(l=10, r=10, t=60, b=60),
)

# Save and display
fig.show()